
# Stage B / NB 10 — Qwen3.5-4B multitask LoRA, five folds (agent A3)

Protocol reference: agent **A3**; experiment families **E5** and **E8**; research question
**RQ1** — *does medical pretraining help at matched scale?*

## This notebook is deliberately a near-copy of NB 09

RQ1 is only a medical-specialization test if MedGemma and Qwen differ in **pretraining and
nothing else**. Every structural choice here is inherited from NB 09: the same folds, the same
task composition, the same response-only collator design, the same selection criterion, the same
token-probability scoring, the same invalid-output policy, and the same resume safeguards.

By default it also inherits NB 09's **selected hyperparameters** (`INHERIT_NB09_CONFIG = True`),
read from `nb09_medgemma_lora/sweep_selection.json`. Running its own sweep would optimise Qwen
while MedGemma keeps a configuration chosen under a shorter schedule, and the resulting
comparison would confound pretraining with tuning effort.

## Where the architectures force a difference

These are recorded in `run_config.json` under `architecture_deviations`, and every one of them
must be disclosed in the manuscript rather than presented as "identical settings".

| difference | why it is unavoidable |
| --- | --- |
| `AutoModelForMultimodalLM` instead of `AutoModelForMultimodalLM` | Qwen3.5's loader class |
| visual-token budget (`min_pixels`/`max_pixels`, 256–1024 tokens) | Qwen3.5 uses dynamic resolution; MedGemma has a fixed grid |
| LoRA target discovery by module scan, not a fixed projection list | Qwen3.5 mixes full-attention and linear-attention text layers, so `q/k/v/o` alone would leave much of the backbone untouched |
| `enable_thinking=False` | Qwen3.5 reasons by default; direct JSON supervision must not contain a `<think>` trace |

Trainable-parameter counts will therefore differ between the two arms. **Report both.** A win
that comes with 3x the trainable parameters is a scale result, not a specialization result.

## Carried over byte-for-byte from the tested notebook

`supervised_messages`, `HarmonyImageTextDataset`, `MultiModalResponseOnlyCollator`,
`render_chat_template`, and `discover_qwen_text_lora_targets` are copied verbatim from
`finetune_qwen35_4b_multitask_lora_rank32_5fold.ipynb`. They handle Qwen's pixel budget and
hybrid attention correctly and are known to train.

## Outputs (under `stage_B/nb10_qwen_lora/`)
Same layout as NB 09, plus `rq1_comparison.csv`.


## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

In [ ]:
from importlib.metadata import version

from huggingface_hub import notebook_login, whoami
from peft import LoraConfig, get_peft_model
from transformers.trainer_utils import get_last_checkpoint
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoModelForMultimodalLM, AutoProcessor, StoppingCriteria, StoppingCriteriaList,
    Trainer, TrainingArguments,
)

Image.MAX_IMAGE_PIXELS = None
for package in ["torch", "transformers", "peft", "accelerate", "huggingface-hub"]:
    try:
        print(f"  {package}: {version(package)}")
    except Exception as exc:
        print(f"  {package}: unavailable ({exc})")

# MedGemma is gated; the token is stored by the HF client, not written into the notebook.
try:
    identity = whoami()
except Exception:
    notebook_login()
    identity = whoami()
print("Logged in as:", identity.get("name"))

if not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
    raise RuntimeError("A BF16-capable CUDA GPU is required for this notebook.")

# ---- Environment compatibility shims -----------------------------------------------------
import platform as _platform

def model_load_kwargs(dtype=torch.bfloat16, **extra):
    """
    `torch_dtype` was renamed to `dtype` in recent transformers. Passing the old name still
    works today but emits a deprecation warning and will eventually break, so pick whichever
    the installed version accepts rather than pinning to one.
    """
    import inspect
    from transformers import PreTrainedModel
    try:
        signature = inspect.signature(PreTrainedModel.from_pretrained)
        key = "dtype" if "dtype" in signature.parameters else "torch_dtype"
    except Exception:
        key = "torch_dtype"
    return {key: dtype, **extra}


def _kernel_tuple():
    try:
        release = _platform.release().split("-")[0]
        return tuple(int(part) for part in release.split(".")[:2])
    except Exception:
        return (99, 99)


KERNEL = _kernel_tuple()
OLD_KERNEL = KERNEL < (5, 5)
if OLD_KERNEL:
    # accelerate warns that kernels below 5.5 "can cause the process to hang". The hang is in
    # DataLoader worker processes, and it is not hypothetical: it is the classic way a
    # multi-day training job dies silently overnight with no traceback. Single-process loading
    # is slower per step but cannot deadlock, which is the right trade for a 65 GPU-hour run.
    SAFE_DATALOADER_WORKERS = 0
    SAFE_PIN_MEMORY = False
    print(f"Kernel {_platform.release()} is below 5.5 -> forcing "
          f"dataloader_num_workers=0 and pin_memory=False to avoid worker hangs.")
    print("  Set OVERRIDE_OLD_KERNEL_GUARD = True in the config cell to opt out.")
else:
    SAFE_DATALOADER_WORKERS = None   # keep the configured value
    SAFE_PIN_MEMORY = True


## 2. Configuration

In [ ]:
import re

NB10_DIR = STAGE_B_DIR / "nb10_qwen_lora"
SWEEP_DIR = NB10_DIR / "sweeps"
FOLD_DIR = NB10_DIR / "folds"
for directory in [NB10_DIR, SWEEP_DIR, FOLD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_REVISION = MODEL_REVISIONS.get(MODEL_ID)
AGENT_NAME = "A3_qwen_lora"

# ---- Baseline configuration: the tested notebook's settings, unchanged --------------------
BASE_CONFIG = {
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_policy": "qwen_text_backbone",  # module scan; see the header table
    "learning_rate": 1e-4,
    "warmup_ratio": 0.05,
    "epochs": 5,
    "composition": "M5",                    # mRALE + COVID + truthfulness auxiliaries
    "imbalance": "E8a",                     # as-is
}
MAX_LENGTH = 2048

# Qwen3.5 uses 16-pixel vision patches with 2x2 spatial merging, so one language-model image
# token covers roughly 32x32 input pixels. Cap each CXR at 1,024 visual tokens, leaving at least
# 1,024 positions for the instructions and the supervised JSON response. Carried over from the
# tested notebook.
QWEN_VISION_PIXEL_FACTOR = 16 * 2
MIN_IMAGE_TOKENS, MAX_IMAGE_TOKENS = 256, 1024
MIN_IMAGE_PIXELS = MIN_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR ** 2
MAX_IMAGE_PIXELS = MAX_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR ** 2

# Qwen3.5 thinks by default. Direct JSON supervision and scoring must not include a <think>
# trace, matching the tested notebook's DISABLE_THINKING.
DISABLE_THINKING = True

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
DATALOADER_NUM_WORKERS = 2
# Set True to keep the configured worker count on a pre-5.5 kernel despite
# the documented hang risk. Not recommended for a multi-day run.
OVERRIDE_OLD_KERNEL_GUARD = False

# ---- Phase A: sweeps, fold 0 only ---------------------------------------------------------
# Sweeping all five folds would cost ~5x for no additional selection information, and the
# protocol requires selection on inner validation anyway. The sweep is reported as fold-0
# inner-validation evidence and the manuscript says so.
# RQ1 requires matched tuning effort, so by default this arm INHERITS NB 09's selected
# configuration rather than running its own sweep. Set INHERIT_NB09_CONFIG = False and
# RUN_SWEEPS = True only if you intend to report a tuned-vs-tuned comparison, and say so.
INHERIT_NB09_CONFIG = True
NB09_DIR = STAGE_B_DIR / "nb09_medgemma_lora"

RUN_SWEEPS = False
SWEEP_FOLD = 0
SWEEP_EPOCHS = 2            # short runs are enough to rank configurations
SWEEPS = {
    "E5-R_rank":      [{"lora_r": r, "lora_alpha": 2 * r} for r in [8, 16, 32, 64]],
    "E5-T_targets":   [{"target_policy": p} for p in
                       ["language_attn_only", "language_attn_mlp",
                        "language_attn_mlp_projector", "all_linear_incl_vision"]],
    "E5-L_optimizer": [{"learning_rate": lr, "warmup_ratio": wr}
                       for lr in [5e-5, 1e-4, 3e-4] for wr in [0.03, 0.05, 0.10]],
    "E5-M_composition": [{"composition": c} for c in ["M1", "M2", "M3", "M5"]],
    "E8_imbalance":   [{"imbalance": a} for a in ["E8a", "E8b", "E8e"]],
}
RUN_SWEEP_FAMILIES = list(SWEEPS)

# ---- Phase B: the selected configuration on all five folds --------------------------------
RUN_FINAL_FOLDS = True
FINAL_FOLDS = [0, 1, 2, 3, 4]
# Set to a dict to override sweep selection, e.g. {"lora_r": 32, "learning_rate": 1e-4}
FORCE_FINAL_CONFIG = None

# ---- Evaluation ---------------------------------------------------------------------------
RUN_GENERATION_EVAL = True
RUN_TOKEN_SCORING = True          # protocol 7.2; without it this arm has no AUROC
USE_JSON_STOP_CRITERIA = True     # E6-Q
GENERATION_MAX_NEW_TOKENS = {"covid_classification": 48, "mrale_prediction": 256}
RUN_EXTERNAL = True

SMOKE_TEST = False                # True -> 1 fold, 0.02 epochs, 16 eval images

# The rejected submission's numbers, kept only so the response letter can show the correction.
SUPERSEDED_LEAKY_REFERENCE = {
    "source": "qwen35_4b_multitask_lora_rank32_cv (legacy multi_task_CV folds), if it exists",
    "mrale_mae": 3.880, "mrale_mae_ci95": [3.498, 4.262],
    "covid_balanced_accuracy": 0.616, "covid_specificity": 0.413,
    "status": "INVALID -- 135-153 study groups appeared in both train and test per fold",
    "expected_direction_on_clean_folds": "worse; leakage inflated these figures",
}

if SMOKE_TEST:
    RUN_SWEEPS, FINAL_FOLDS = False, [0]
    BASE_CONFIG = {**BASE_CONFIG, "epochs": 0.02}

print("Model:", MODEL_ID, "| revision:", MODEL_REVISION)
print("Baseline config:", json.dumps(BASE_CONFIG, indent=2))
print("Sweep families:", RUN_SWEEP_FAMILIES if RUN_SWEEPS else "disabled")
print("Output:", NB10_DIR)

# ---- Inherit NB 09's selected configuration (RQ1 matching) --------------------------------
# Refuse to start unmatched rather than merely warn; see the raise below.
ALLOW_UNMATCHED_CONFIG = False

INHERITED_FROM_NB09 = None
if INHERIT_NB09_CONFIG:
    # Reads NB 09's selection. (NB 10 writes its OWN sweep_selection.json later, to NB10_DIR;
    # these are different files and conflating them is what broke this the first time.)
    selection_path = NB09_DIR / "sweep_selection.json"
    if selection_path.is_file():
        chosen = json.loads(selection_path.read_text(encoding="utf-8")).get("final_config", {})
        for key in ["lora_r", "lora_alpha", "lora_dropout", "learning_rate",
                    "warmup_ratio", "epochs", "composition", "imbalance"]:
            if key in chosen:
                BASE_CONFIG[key] = chosen[key]
        INHERITED_FROM_NB09 = {k: BASE_CONFIG[k] for k in
                               ["lora_r", "lora_alpha", "learning_rate", "warmup_ratio",
                                "epochs", "composition", "imbalance"]}
        print("Inherited NB 09's selected configuration for RQ1 matching:")
        print(json.dumps(INHERITED_FROM_NB09, indent=2))
        print("Target-module policy is NOT inherited: the architectures differ (see header).")
    elif ALLOW_UNMATCHED_CONFIG:
        print(f"WARNING: {selection_path} not found. Proceeding with this notebook's own "
              "BASE_CONFIG because ALLOW_UNMATCHED_CONFIG is True. RQ1 will then compare two "
              "DIFFERENTLY TUNED models, and the manuscript must say so rather than present "
              "it as a matched-scale specialization test.")
    else:
        # A printed warning is not enough protection for a 55 GPU-hour commitment. Stopping
        # here costs a minute; discovering it afterwards costs the run, because the result
        # would not answer RQ1.
        searched = [selection_path,
                    STAGE_B_DIR / "nb09_medgemma_lora" / "sweep_selection.json",
                    STAGE_ROOT / "stage_B" / "nb09_medgemma_lora" / "sweep_selection.json"]
        existing = [str(path) for path in searched if path.is_file()]
        if existing:
            raise RuntimeError(
                f"{selection_path} not found, but a selection file DOES exist at {existing}. "
                "That is a path-configuration mismatch, not a missing NB 09 run: point "
                "NB09_DIR at the right directory and re-run this cell."
            )
        raise RuntimeError(
            f"{selection_path} not found (also checked {[str(x) for x in searched[1:]]}), "
            "so this arm cannot inherit NB 09's selected "
            "configuration.\n"
            "RQ1 is a medical-specialization test only if the two arms differ in pretraining "
            "and not in tuning effort, so training now would spend ~55 GPU-hours on a result "
            "that cannot answer it.\n"
            "Options:\n"
            "  1. Wait for NB 09 PHASE A only (the sweeps, ~25 GPU-h). All of NB 09 is NOT "
            "required: sweep_selection.json is written as soon as selection completes.\n"
            "  2. INHERIT_NB09_CONFIG = False to run Qwen with its own sweeps and report a "
            "tuned-vs-tuned comparison.\n"
            "  3. ALLOW_UNMATCHED_CONFIG = True to proceed with BASE_CONFIG and disclose the "
            "mismatch."
        )
print()
print("Model:", MODEL_ID, "| revision:", MODEL_REVISION)
print("Config:", json.dumps(BASE_CONFIG, indent=2))


In [ ]:
# Shared evaluation helpers, identical to those used by NB 05-08 so that every arm in
# Table 2 is scored by the same code path.
def evaluate_arm(rows, label):
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")


print("Evaluation helpers ready.")


## 3. Load the regenerated folds

`FOLD_DATA_DIR` points at Stage A NB 02's output, not at `multi_task_CV`. The task set is
**auto-detected** from what NB 02 actually wrote rather than hardcoded, so a change to the
fold builder cannot silently desynchronise from this notebook.

In [ ]:
FOLD_DATA_DIR = FOLD_DEF_DIR
TRAIN_PATTERN = "multitask_train_fold_{fold}_harmony.jsonl"
TEST_PATTERN = "multitask_test_fold_{fold}_harmony.jsonl"

for fold in range(N_FOLDS):
    for pattern in [TRAIN_PATTERN, TEST_PATTERN]:
        path = FOLD_DATA_DIR / pattern.format(fold=fold)
        if not path.is_file():
            raise FileNotFoundError(
                f"{path} not found. Run Stage A NB 02 first. Do NOT point this notebook at "
                "/data/liangz2/openi/midrc/multi_task_CV: those folds leak at study level and "
                "every number computed on them is superseded."
            )


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{number}: {exc}") from exc
    return rows


def answer_text(record):
    ground_truth = record.get("ground_truth", {})
    if isinstance(ground_truth, dict) and ground_truth.get("answer") is not None:
        return str(ground_truth["answer"]).strip()
    for message in reversed(record["messages"]):
        if message.get("role") == "assistant":
            content = message.get("content", "")
            if isinstance(content, str):
                return content.strip()
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    return str(item["text"]).strip()
    raise ValueError(f"No assistant target in record {record.get('id')}")


fold_records = {}
observed_tasks = Counter()
for fold in range(N_FOLDS):
    train = read_jsonl(FOLD_DATA_DIR / TRAIN_PATTERN.format(fold=fold))
    test = read_jsonl(FOLD_DATA_DIR / TEST_PATTERN.format(fold=fold))
    fold_records[fold] = {"train": train, "test": test}
    observed_tasks.update(record["task"] for record in train)

ALL_TASKS = sorted(observed_tasks)
DIRECT_TASKS = sorted(task for task in ALL_TASKS if "truthfulness" not in task)
AUXILIARY_TASKS = sorted(task for task in ALL_TASKS if "truthfulness" in task)

print("Tasks present in the regenerated folds:")
for task, count in observed_tasks.most_common():
    kind = "auxiliary" if task in AUXILIARY_TASKS else "DIRECT"
    print(f"  {task:<32} {count:>7,}  [{kind}]")
print()
print("DIRECT_TASKS   :", DIRECT_TASKS)
print("AUXILIARY_TASKS:", AUXILIARY_TASKS)
if "normality_classification" in ALL_TASKS:
    print()
    print("WARNING: normality_classification is present. Protocol 3.3 holds Montgomery out "
          "entirely as cohort X1, so it should have no training data. Check NB 02's "
          "INCLUDE_MONTGOMERY_IN_TRAINING flag.")

for fold in range(N_FOLDS):
    train_images = {record["image_path"] for record in fold_records[fold]["train"]}
    test_images = {record["image_path"] for record in fold_records[fold]["test"]}
    overlap = train_images & test_images
    if overlap:
        raise RuntimeError(
            f"Fold {fold}: {len(overlap)} images appear in both train and test. NB 02's gate "
            "should have caught this; do not train until it is resolved."
        )
print()
print("Leakage re-check passed: no image appears in both train and test of any fold.")

## 4. Multitask composition (E5-M) and imbalance (E8) filters

**E5-M** asks whether multitask coupling transfers or trades one task against the other
(RQ5). M5 is the tested configuration; M1/M2 isolate the single-task baselines.

**E8** attacks the specificity collapse. At 93.5% PCR prevalence the model can score 0.93
accuracy by answering "Yes" every time, which is close to what the rejected version did
(sensitivity 0.818, specificity 0.413). Balanced sampling changes what the model sees;
threshold tuning (E8d) changes how its output is read and is applied at evaluation time.

In [ ]:
COMPOSITIONS = {
    "M1": {"tasks": ["mrale_prediction"], "note": "mRALE only"},
    "M2": {"tasks": ["covid_classification"], "note": "COVID only"},
    "M3": {"tasks": ["mrale_prediction", "covid_classification"], "note": "both direct tasks"},
    "M5": {"tasks": None, "note": "all direct tasks + truthfulness auxiliaries (tested config)"},
}


def apply_composition(records, composition):
    keep = COMPOSITIONS[composition]["tasks"]
    if keep is None:
        return list(records)
    return [record for record in records if record["task"] in set(keep)]


def label_stratum(record):
    task = record["task"]
    target = answer_text(record)
    if task == "covid_classification":
        try:
            return json.loads(target).get("covid_positive")
        except Exception:
            return None
    if task in AUXILIARY_TASKS:
        return target
    if task == "mrale_prediction":
        try:
            return cm.severity_band(int(json.loads(target)["mRALE Score"]))
        except Exception:
            return None
    return None


def apply_imbalance(records, arm, seed):
    """
    E8a  as-is
    E8b  class-balanced sampling of covid_classification records
    E8e  severity-band-balanced sampling of mrale_prediction records
    E8c/E8d are loss reweighting and threshold tuning; they act elsewhere (see below).
    """
    if arm == "E8a":
        return list(records)
    rng = random.Random(seed)
    by_task = defaultdict(list)
    for record in records:
        by_task[record["task"]].append(record)

    target_task = "covid_classification" if arm == "E8b" else "mrale_prediction"
    if target_task not in by_task:
        return list(records)

    strata = defaultdict(list)
    for record in by_task[target_task]:
        strata[label_stratum(record)].append(record)
    strata = {key: value for key, value in strata.items() if key is not None}
    if len(strata) < 2:
        return list(records)

    # Oversample the minority strata up to the majority size, with replacement. Undersampling
    # the majority would discard most of a 2,581-image cohort, which is worse here.
    largest = max(len(value) for value in strata.values())
    balanced = []
    for key in sorted(strata, key=str):
        items = strata[key]
        balanced.extend(items)
        while len([item for item in balanced if label_stratum(item) == key]) < largest:
            balanced.append(rng.choice(items))

    result = [record for record in records if record["task"] != target_task] + balanced
    rng.shuffle(result)
    return result


def grouped_inner_split_records(records, fraction, seed):
    # Group by MIDRC study directory, matching the tested notebook and NB 02's grouping unit.
    def group_of(record):
        return str(Path(record["image_path"]).parent)

    groups = sorted({group_of(record) for record in records})
    rng = random.Random(seed)
    rng.shuffle(groups)
    validation_groups = set(groups[:max(1, round(len(groups) * fraction))])
    train = [record for record in records if group_of(record) not in validation_groups]
    validation = [record for record in records if group_of(record) in validation_groups]
    assert not ({group_of(r) for r in train} & {group_of(r) for r in validation})
    return train, validation


def prepare_fold(fold, config):
    train_all = fold_records[fold]["train"]
    train_all = apply_composition(train_all, config["composition"])
    optimisation, validation = grouped_inner_split_records(
        train_all, INTERNAL_VALIDATION_FRACTION, SEED + fold)
    optimisation = apply_imbalance(optimisation, config["imbalance"], SEED + 1000 + fold)
    test = [record for record in fold_records[fold]["test"] if record["task"] in DIRECT_TASKS]
    return optimisation, validation, test


for name, spec in COMPOSITIONS.items():
    sample = apply_composition(fold_records[0]["train"], name)
    print(f"  {name}: {len(sample):>7,} records  ({spec['note']})")
print()
for arm in ["E8a", "E8b", "E8e"]:
    sample = apply_imbalance(
        apply_composition(fold_records[0]["train"], "M5"), arm, SEED)
    covid = [r for r in sample if r["task"] == "covid_classification"]
    balance = Counter(label_stratum(r) for r in covid)
    print(f"  {arm}: {len(sample):>7,} records | covid label balance {dict(balance)}")

## 5. Dataset and response-only collator

Carried over **verbatim** from the tested notebook. Only assistant response tokens contribute
to the loss; prompt and padding tokens are masked with -100. The image is inserted at the first
user turn. This code is known to work and is deliberately not "improved" here — changing it
would make the E5 sweep results incomparable with the tested baseline.

In [ ]:
# ---------------------------------------------------------------------------------
# CARRIED OVER BYTE-FOR-BYTE from the tested notebook
# finetune_qwen35_4b_multitask_lora_rank32_5fold.ipynb (cells 8 and 10).
# Do not reformat. This code handles Qwen's visual-token budget and thinking-mode
# suppression correctly and is known to train.
# ---------------------------------------------------------------------------------
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_IMAGE_PIXELS, max_pixels=MAX_IMAGE_PIXELS,
    **({"revision": MODEL_REVISION} if MODEL_REVISION else {}))
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def render_chat_template(processor, messages, add_generation_prompt):
    template_kwargs = {
        "add_generation_prompt": add_generation_prompt,
        "tokenize": False,
    }
    if DISABLE_THINKING:
        template_kwargs["enable_thinking"] = False
    return processor.apply_chat_template(messages, **template_kwargs)


def load_cxr_with_pixel_budget(path):
    """Load a CXR and reduce its area without changing its aspect ratio."""
    with Image.open(path) as image_file:
        image = image_file.convert("RGB")

    width, height = image.size
    area = width * height
    if area <= MAX_IMAGE_PIXELS:
        return image

    scale = math.sqrt(MAX_IMAGE_PIXELS / area)
    resized_width = max(
        QWEN_VISION_PIXEL_FACTOR,
        int(width * scale) // QWEN_VISION_PIXEL_FACTOR * QWEN_VISION_PIXEL_FACTOR,
    )
    resized_height = max(
        QWEN_VISION_PIXEL_FACTOR,
        int(height * scale) // QWEN_VISION_PIXEL_FACTOR * QWEN_VISION_PIXEL_FACTOR,
    )
    return image.resize(
        (resized_width, resized_height),
        resample=Image.Resampling.LANCZOS,
    )


def supervised_messages(record, include_assistant=True):
    messages = []
    image_inserted = False
    for message in record["messages"]:
        role = message["role"]
        if role == "assistant":
            continue
        content = []
        raw_content = message.get("content", "")
        if isinstance(raw_content, str):
            raw_content = [{"type": "text", "text": raw_content}]
        for item in raw_content:
            if isinstance(item, str):
                item = {"type": "text", "text": item}
            if item.get("type") == "text":
                if role == "user" and not image_inserted:
                    content.append({"type": "image"})
                    image_inserted = True
                content.append({"type": "text", "text": item["text"]})
        messages.append({"role": role, "content": content})
    if not image_inserted:
        raise ValueError(f"No user turn available for image in {record.get('id')}")
    if include_assistant:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": answer_text(record)}],
        })
    return messages


class HarmonyImageTextDataset(Dataset):
    def __init__(self, records, processor, max_length):
        self.records = records
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        prompt_messages = supervised_messages(record, include_assistant=False)
        full_messages = supervised_messages(record, include_assistant=True)
        prompt_text = render_chat_template(
            self.processor,
            prompt_messages,
            add_generation_prompt=True,
        )
        full_text = render_chat_template(
            self.processor,
            full_messages,
            add_generation_prompt=False,
        )

        image = load_cxr_with_pixel_budget(record["image_path"])
        full = self.processor(
            text=full_text,
            images=image,
            return_tensors="pt",
            truncation=False,
        )
        prompt = self.processor(
            text=prompt_text,
            images=image,
            return_tensors="pt",
            truncation=False,
        )

        full_length = int(full["input_ids"].shape[-1])
        prompt_length_unclipped = int(prompt["input_ids"].shape[-1])
        if full_length > self.max_length:
            raise ValueError(
                f"Complete sequence for {record['id']} has {full_length} tokens, "
                f"exceeding MAX_LENGTH={self.max_length}. Do not enable multimodal "
                "truncation. Reduce MAX_IMAGE_TOKENS or increase MAX_LENGTH."
            )
        if prompt_length_unclipped >= full_length:
            raise ValueError(
                f"No assistant target tokens remain for {record['id']}: "
                f"prompt={prompt_length_unclipped}, full={full_length}."
            )

        item = {key: value.squeeze(0) for key, value in full.items()}
        labels = item["input_ids"].clone()
        prompt_length = min(prompt["input_ids"].shape[-1], labels.shape[-1])
        labels[:prompt_length] = -100
        if torch.all(labels == -100):
            raise ValueError(
                f"Target was fully truncated for {record['id']}; increase MAX_LENGTH above {self.max_length}."
            )
        item["labels"] = labels
        return item


class MultiModalResponseOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        tokenizer = processor.tokenizer
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        self.pad_token_id = tokenizer.pad_token_id

    @staticmethod
    def _pad_sequence(tensor, target_length, value):
        if tensor.shape[0] == target_length:
            return tensor
        pad_shape = (target_length - tensor.shape[0],) + tuple(tensor.shape[1:])
        padding = torch.full(pad_shape, value, dtype=tensor.dtype)
        return torch.cat([tensor, padding], dim=0)

    def __call__(self, features):
        batch = {}
        sequence_keys = {
            "input_ids",
            "attention_mask",
            "token_type_ids",
            "mm_token_type_ids",
            "labels",
        }
        max_length = max(feature["input_ids"].shape[0] for feature in features)
        for key in sequence_keys:
            if all(key in feature for feature in features):
                value = -100 if key == "labels" else (self.pad_token_id if key == "input_ids" else 0)
                batch[key] = torch.stack([
                    self._pad_sequence(feature[key], max_length, value) for feature in features
                ])

        visual_patch_keys = {"pixel_values", "pixel_values_videos"}
        visual_grid_keys = {"image_grid_thw", "video_grid_thw"}

        common_keys = set.intersection(*(set(feature) for feature in features))
        for key in sorted(common_keys - sequence_keys):
            values = [feature[key] for feature in features]
            shapes = [tuple(value.shape) for value in values]
            try:
                if key in visual_patch_keys:
                    # Qwen stores a variable number of flattened visual patches
                    # for each image. The model expects all patches concatenated.
                    batch[key] = torch.cat(values, dim=0)
                elif key in visual_grid_keys:
                    # __getitem__ removes the singleton image dimension, so
                    # stacking restores one (t, h, w) grid row per image.
                    batch[key] = torch.stack(values, dim=0)
                else:
                    batch[key] = torch.stack(values, dim=0)
            except RuntimeError as exc:
                raise RuntimeError(
                    f"Cannot combine processor field {key}; shapes={shapes}. "
                    "Inspect the Qwen processor output for this field."
                ) from exc
        return batch

print("Dataset and collator ready (byte-identical to the tested Qwen notebook).")
print(f"Visual-token budget: {MIN_IMAGE_TOKENS}-{MAX_IMAGE_TOKENS} tokens "
      f"({MIN_IMAGE_PIXELS:,}-{MAX_IMAGE_PIXELS:,} pixels)")

# ---- Dataset preflight -------------------------------------------------------------------
# Cheap insurance before a 65 GPU-hour run. Build one real batch and assert that (a) image
# tensors are present, (b) the response-only mask left some supervised tokens, and (c) the
# decoded target is the JSON we expect. A silent failure in any of these would train a
# language-only model on masked-out labels and look like a bad result rather than a bug.
_probe_records = [record for record in fold_records[0]["train"]
                  if record["task"] in DIRECT_TASKS][:2]
if _probe_records:
    _probe_dataset = HarmonyImageTextDataset(_probe_records, processor, MAX_LENGTH)
    _batch = MultiModalResponseOnlyCollator(processor)(
        [_probe_dataset[0], _probe_dataset[1 % len(_probe_records)]])
    _vision_keys = [key for key in _batch
                    if key in {"pixel_values", "image_grid_thw", "token_type_ids"}]
    _supervised = int((_batch["labels"] != -100).sum())
    print()
    print("Dataset preflight")
    print(f"  batch keys        : {sorted(_batch)}")
    print(f"  vision tensors    : {_vision_keys or 'NONE'}")
    print(f"  supervised tokens : {_supervised}")
    _decoded = processor.tokenizer.decode(
        [t for t in _batch['labels'][0].tolist() if t != -100], skip_special_tokens=True)
    print(f"  decoded target[0] : {_decoded[:110]!r}")
    if not any(key in _batch for key in ("pixel_values", "image_grid_thw")):
        raise RuntimeError(
            "Dataset preflight FAILED: the collated batch carries no image tensor, so training "
            "would fit a text-only model while appearing to work. Check that "
            "supervised_messages() inserts {'type': 'image'} and that the processor consumes it."
        )
    if _supervised == 0:
        raise RuntimeError(
            "Dataset preflight FAILED: every label is -100, so the loss has no target. The "
            "prompt/response boundary is mis-computed; check MAX_LENGTH and the chat template."
        )
    print("  OK -- images bound and supervision present.")


## 6. LoRA construction with a configurable target policy (E5-T)

The tested notebook attached LoRA to language-decoder attention **and** MLP projections. E5-T
asks whether that choice mattered. Four policies are compared, from attention-only to
everything including the vision tower.

Each policy records its trainable-parameter count, which is what makes the sweep interpretable:
a policy that wins by touching 4x more parameters has not shown that its *placement* is better.

In [ ]:
VISION_NAME_MARKERS = (
    "visual",
    "vision_model",
    "vision_tower",
    "vision_encoder",
)


TEXT_NAME_MARKERS = (
    "language_model",
    "text_model",
    "model.layers",
)


LORA_EXCLUDE_MARKERS = (
    "lm_head",
    "embed_tokens",
    "embedding",
    "mtp",
)


def discover_qwen_text_lora_targets(model):
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue

        lower_name = name.lower()
        if any(marker in lower_name for marker in VISION_NAME_MARKERS):
            continue
        if any(marker in lower_name for marker in LORA_EXCLUDE_MARKERS):
            continue
        if any(marker in lower_name for marker in TEXT_NAME_MARKERS):
            targets.append(name)

    targets = sorted(set(targets))
    if not targets:
        sample_linear_names = [
            name for name, module in model.named_modules()
            if isinstance(module, torch.nn.Linear)
        ][:30]
        raise RuntimeError(
            "No Qwen text-backbone LoRA targets were discovered. "
            f"Example linear module names: {sample_linear_names}"
        )
    return targets



def discover_targets(model, policy):
    """
    Qwen3.5 mixes full-attention and linear-attention text layers, so a fixed q/k/v/o list would
    leave much of the backbone unadapted. The tested notebook's module scan is used instead;
    `policy` selects how wide that scan goes, which is what E5-T varies.
    """
    targets = discover_qwen_text_lora_targets(model)
    if policy == "qwen_text_attention_only":
        targets = [name for name in targets
                   if name.rsplit(".", 1)[-1] in {"q_proj", "k_proj", "v_proj", "o_proj"}]
    elif policy == "qwen_text_backbone":
        pass                       # the tested default: every text-backbone linear layer
    elif policy == "qwen_text_backbone_plus_projector":
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear) and any(
                    marker in name for marker in
                    ("visual.merger", "multi_modal_projector", "mm_projector")):
                targets.append(name)
    else:
        raise ValueError(f"Unknown Qwen target policy {policy}")
    targets = sorted(set(targets))
    if not targets:
        raise RuntimeError(f"Policy '{policy}' matched no modules.")
    return targets


def build_lora_model(config):
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, **model_load_kwargs(
            torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
            **({"revision": MODEL_REVISION} if MODEL_REVISION else {})))
    model.config.use_cache = False
    # The generation config carries its own copy, which is what transformers warns
    # about when gradient checkpointing is enabled. Harmless, but silencing it keeps
    # a genuinely important warning from being lost in the noise of a long log.
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    targets = discover_targets(model, config["target_policy"])
    peft_config = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"], bias="none",
        task_type="CAUSAL_LM", target_modules=targets)
    model = get_peft_model(model, peft_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    stats = {"target_policy": config["target_policy"], "n_target_modules": len(targets),
             "trainable_parameters": int(trainable), "total_parameters": int(total),
             "trainable_fraction": round(trainable / total, 6),
             "example_targets": targets[:6]}
    print(f"  LoRA r={config['lora_r']} policy={config['target_policy']}: "
          f"{len(targets)} modules, {trainable / 1e6:.2f}M trainable "
          f"({trainable / total:.4%})")
    print("  NOTE: compare this against NB 09's count. A win with far more trainable "
          "parameters is a scale result, not a specialization result.")
    return model, stats


def release(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


## 7. Generation evaluation with token-probability scoring

The same two-pass design as NB 07, and the scoring functions are intentionally identical: a
zero-shot AUROC computed by sequence likelihood is not comparable to a LoRA AUROC computed some
other way, so both notebooks must use one method.

Also carries the **E6-Q** fix — generation stops at the first closed JSON object.

In [ ]:
DIGIT_RANGES = {"extent_right_numerical": 5, "density_right_numerical": 4,
                "extent_left_numerical": 5, "density_left_numerical": 4}


class BalancedJsonStop(StoppingCriteria):
    """E6-Q: halt at the first complete top-level JSON object."""

    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0, self.prompt_length:],
                                     skip_special_tokens=True)
        start = text.find("{")
        if start < 0:
            return False
        depth, in_string, escape = 0, False, False
        for character in text[start:]:
            if escape:
                escape = False
                continue
            if character == "\\":
                escape = True
                continue
            if character == '"':
                in_string = not in_string
                continue
            if in_string:
                continue
            if character == "{":
                depth += 1
            elif character == "}":
                depth -= 1
                if depth == 0:
                    return True
        return False


def model_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return torch.device("cuda")


def prepare_inputs(prompt_text, image_path, target_device):
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
        inputs = processor(text=prompt_text, images=image, return_tensors="pt")
    return {key: (value.to(device=target_device, dtype=torch.bfloat16)
                  if value.is_floating_point() else value.to(target_device))
            for key, value in inputs.items()}


def prompt_for(record):
    # render_chat_template carries the tested notebook's enable_thinking handling, so
    # generation and scoring both see a prompt with no <think> trace.
    return render_chat_template(
        processor, supervised_messages(record, include_assistant=False),
        add_generation_prompt=True)


@torch.inference_mode()
def generate_record(model, record):
    prompt_text = prompt_for(record)
    target_device = model_device(model)
    inputs = prepare_inputs(prompt_text, record["image_path"], target_device)
    prompt_length = inputs["input_ids"].shape[-1]
    stopping = (StoppingCriteriaList([BalancedJsonStop(processor.tokenizer, prompt_length)])
                if USE_JSON_STOP_CRITERIA else None)
    output_ids = model.generate(
        **inputs, do_sample=False,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS.get(record["task"], 256),
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        stopping_criteria=stopping, use_cache=True)
    completion = output_ids[0, prompt_length:]
    return (processor.decode(completion, skip_special_tokens=True).strip(),
            int(completion.shape[-1]))


@torch.inference_mode()
def sequence_log_likelihood(model, record, candidate_text):
    prompt_text = prompt_for(record)
    target_device = model_device(model)
    full = prepare_inputs(prompt_text + candidate_text, record["image_path"], target_device)
    prompt_only = prepare_inputs(prompt_text, record["image_path"], target_device)
    prompt_length = prompt_only["input_ids"].shape[-1]
    total_length = full["input_ids"].shape[-1]
    if total_length <= prompt_length:
        return float("nan")
    logits = model(**full).logits.float()
    log_probs = torch.log_softmax(logits[0, prompt_length - 1:total_length - 1], dim=-1)
    targets = full["input_ids"][0, prompt_length:total_length]
    return float(log_probs.gather(-1, targets[:, None]).squeeze(-1).sum())


def covid_probability(model, record):
    scores = {}
    for value in ["Yes", "No"]:
        candidate = json.dumps({"covid_positive": value}, separators=(",", ":"))
        scores[value] = sequence_log_likelihood(model, record, candidate)
    yes, no = scores["Yes"], scores["No"]
    if math.isnan(yes) or math.isnan(no):
        return None, scores
    maximum = max(yes, no)
    probability = math.exp(yes - maximum) / (math.exp(yes - maximum) + math.exp(no - maximum))
    return probability, scores


def extract_json_object(text):
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(obj, dict):
        raise ValueError("Not a JSON object")
    return obj


def count_json_objects(text):
    count, depth, in_string, escape = 0, 0, False, False
    for character in text:
        if escape:
            escape = False
            continue
        if character == "\\":
            escape = True
            continue
        if character == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                count += 1
    return count


def mrale_from_parsed(parsed):
    if not isinstance(parsed, dict):
        return None
    values = {}
    for field, n_classes in DIGIT_RANGES.items():
        try:
            value = int(parsed.get(field))
        except (TypeError, ValueError):
            return None
        if not (0 <= value < n_classes):
            return None
        values[field] = value
    right = values["extent_right_numerical"] * values["density_right_numerical"]
    left = values["extent_left_numerical"] * values["density_left_numerical"]
    try:
        reported = int(parsed.get("mRALE Score"))
    except (TypeError, ValueError):
        reported = None
    return {
        "extent_right": values["extent_right_numerical"],
        "density_right": values["density_right_numerical"],
        "extent_left": values["extent_left_numerical"],
        "density_left": values["density_left_numerical"],
        "mrale_right": right, "mrale_left": left, "mrale_total": right + left,
        "reported_total": reported, "formula_consistent": reported == right + left,
    }


def ground_truth_from_record(record):
    meta = record.get("meta", {})
    reference = meta.get("reference_mrale") or {}
    covid = meta.get("covid_positive")
    if record["task"] == "covid_classification":
        try:
            covid = json.loads(answer_text(record)).get("covid_positive", covid)
        except Exception:
            pass
    total = reference.get("total")
    if record["task"] == "mrale_prediction":
        try:
            parsed = json.loads(answer_text(record))
            reference = {"er": parsed["extent_right_numerical"],
                         "dr": parsed["density_right_numerical"],
                         "el": parsed["extent_left_numerical"],
                         "dl": parsed["density_left_numerical"],
                         "total": parsed["mRALE Score"]}
            total = reference["total"]
        except Exception:
            pass
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": None if total is None else int(total),
        "gt_mrale_right": (None if not reference else
                           int(reference["er"]) * int(reference["dr"])),
        "gt_mrale_left": (None if not reference else
                          int(reference["el"]) * int(reference["dl"])),
        "gt_extent_right": reference.get("er"), "gt_density_right": reference.get("dr"),
        "gt_extent_left": reference.get("el"), "gt_density_left": reference.get("dl"),
    }

## 8. Evaluate one fold

Generation over the held-out fold, plus token scoring, producing rows in the shared schema.
COVID and mRALE come from separate records for the same image, so they are merged into one row
per image before metrics are computed.

In [ ]:
def evaluate_fold(model, records, fold, arm, max_images=None, checkpoint_dir=None):
    # Generation + token scoring over ~900 held-out images is 1-2 GPU-hours. Checkpointing per
    # (image, task) means an interruption costs one image rather than the whole fold.
    raw_path = Path(checkpoint_dir) / "eval_raw.jsonl" if checkpoint_dir else None
    seen = cm.load_jsonl_by_key(raw_path, ["image_key", "task"]) if raw_path else {}
    if seen:
        print(f"    resuming evaluation with {len(seen):,} cached (image, task) result(s)")
    by_image = {}
    scores = []
    if max_images is not None:
        keep = sorted({record["file_name"] for record in records})[:max_images]
        records = [record for record in records if record["file_name"] in keep]

    for position, record in enumerate(records, start=1):
        image_key = f"MIDRC::{record['file_name']}"
        cached = seen.get((image_key, record["task"]))
        if cached is not None:
            text = cached.get("raw_output") or ""
            n_tokens = cached.get("n_tokens") or 0
            parse_error = cached.get("parse_error")
            cached_score = cached.get("covid_score")
            cached_candidates = cached.get("candidates")
        started = time.perf_counter()
        try:
            if cached is None:
                text, n_tokens = generate_record(model, record)
                parse_error = None
        except Exception as exc:
            text, n_tokens, parse_error = "", 0, f"generation_failed: {type(exc).__name__}: {exc}"
        try:
            parsed = extract_json_object(text) if text else None
            if parsed is None and parse_error is None:
                parse_error = "empty_output"
        except Exception as exc:
            parsed, parse_error = None, f"{type(exc).__name__}: {exc}"

        entry = by_image.setdefault(image_key, {
            "image_key": image_key, "cohort": "MIDRC", "subcohort": "MIDRC",
            "filename": record["file_name"], "held_out_fold": fold,
            "agent": AGENT_NAME, "arm": arm, "view": "v0", "task": "joint",
            "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
            "seconds": 0.0, "extra": {},
            **ground_truth_from_record(record),
        })
        for key, value in ground_truth_from_record(record).items():
            if entry.get(key) is None:
                entry[key] = value
        entry["seconds"] += round(time.perf_counter() - started, 4)
        entry["extra"][f"{record['task']}_n_tokens"] = n_tokens
        entry["extra"][f"{record['task']}_n_json_objects"] = count_json_objects(text)

        if record["task"] == "covid_classification":
            value = None
            if isinstance(parsed, dict):
                raw = str(parsed.get("covid_positive", "")).strip().lower()
                value = "Yes" if raw in {"yes", "true", "1"} else (
                    "No" if raw in {"no", "false", "0"} else None)
            entry["covid_pred"] = value
            entry["extra"]["covid_parse_error"] = parse_error or (
                None if value else "covid_value_not_recognised")
            if RUN_TOKEN_SCORING:
                try:
                    cache_has_candidates = (
                        isinstance(cached_candidates, dict)
                        and {"Yes", "No"}.issubset(cached_candidates)
                    )
                    if cached is not None and cached_score is not None and cache_has_candidates:
                        probability, candidates = cached_score, cached_candidates
                    else:
                        # Old checkpoints sometimes saved the probability without the two
                        # candidate log-likelihoods. Recompute instead of raising KeyError.
                        probability, candidates = covid_probability(model, record)
                        if cached is not None and cached_score is not None:
                            cached = None  # append a repaired last-write-wins cache row below
                    entry["covid_score"] = probability
                    entry["extra"]["candidate_log_likelihoods"] = candidates
                    if probability is not None:
                        scores.append({
                            "arm": arm, "fold": fold, "image_key": image_key,
                            "covid_score": probability,
                            "logL_yes": candidates["Yes"], "logL_no": candidates["No"],
                            "generated_decision": value,
                            "scored_decision": "Yes" if probability >= 0.5 else "No",
                            "gt_covid": entry["gt_covid"],
                        })
                except Exception as exc:
                    entry["extra"]["scoring_error"] = f"{type(exc).__name__}: {exc}"
        else:
            decoded = mrale_from_parsed(parsed)
            entry["extra"]["mrale_parse_error"] = parse_error or (
                None if decoded else "mrale_fields_missing_or_out_of_range")
            if decoded:
                for key in ["mrale_total", "mrale_right", "mrale_left",
                            "extent_right", "density_right", "extent_left", "density_left"]:
                    entry[key] = decoded[key]
                entry["extra"]["reported_total"] = decoded["reported_total"]
                entry["extra"]["formula_consistent"] = decoded["formula_consistent"]

        if raw_path is not None and cached is None:
            cm.append_jsonl(raw_path, {
                "image_key": image_key, "task": record["task"],
                "raw_output": text[:4000], "n_tokens": n_tokens,
                "parse_error": parse_error,
                "covid_score": entry.get("covid_score"),
                "candidates": entry.get("extra", {}).get("candidate_log_likelihoods"),
            })
        if position % 100 == 0:
            print(f"      eval {position}/{len(records)}")

    rows = []
    for entry in by_image.values():
        errors = [entry["extra"].get("covid_parse_error"),
                  entry["extra"].get("mrale_parse_error")]
        errors = [error for error in errors if error]
        entry["parse_error"] = "; ".join(errors) if errors else None
        entry["valid"] = not errors
        rows.append(cm.make_prediction_row(**entry))
    return rows, scores

## 9. Train one fold

`resume_from_checkpoint` is honoured so an interrupted run continues. Checkpoint selection is
by inner-validation loss, matching the tested notebook. The best adapter directory holds LoRA
weights only.

In [ ]:
def build_training_arguments(**kwargs):
    """
    TrainingArguments with a version-tolerant eval-strategy keyword.

    `evaluation_strategy` was renamed to `eval_strategy` in transformers 4.46. Passing the wrong
    one raises TypeError at construction, which would kill a 65 GPU-hour run in its first
    seconds after the queue finally allocated the node. Try the new name, fall back to the old.
    """
    strategy = kwargs.pop("eval_strategy", None)
    if strategy is not None:
        try:
            return TrainingArguments(eval_strategy=strategy, **kwargs)
        except TypeError:
            return TrainingArguments(evaluation_strategy=strategy, **kwargs)
    return TrainingArguments(**kwargs)


def adapter_weights_present(adapter_dir):
    adapter_dir = Path(adapter_dir)
    candidates = [adapter_dir / "adapter_model.safetensors",
                  adapter_dir / "adapter_model.bin"]
    return any(path.is_file() and path.stat().st_size > 0 for path in candidates)


def _finite_number(value):
    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def recover_training_metrics(output_dir, summary=None):
    """Recover durable fold losses after an adapter-only evaluation resume.

    Older runs saved the adapter before generation but did not save validation_loss beside
    it. If generation was interrupted, the resume path rebuilt fold_summary.json without
    that field and the gate mislabeled the missing value as training divergence.
    """
    output_dir = Path(output_dir)
    summary = summary or {}
    if _finite_number(summary.get("validation_loss")):
        return {"validation_loss": float(summary["validation_loss"]),
                "validation_loss_source": summary.get("validation_loss_source",
                                                       "fold_summary")}

    durable = output_dir / "training_metrics.json"
    if durable.is_file():
        try:
            metrics = json.loads(durable.read_text(encoding="utf-8"))
        except Exception:
            metrics = {}
        if _finite_number(metrics.get("validation_loss")):
            return {"validation_loss": float(metrics["validation_loss"]),
                    "validation_loss_source": "training_metrics.json"}

    # Hugging Face writes best_metric into Trainer state. Prefer the latest checkpoint's
    # state; it describes the adapter restored by load_best_model_at_end.
    state_paths = list((output_dir / "trainer").glob("checkpoint-*/trainer_state.json"))
    def checkpoint_step(path):
        try:
            return int(path.parent.name.rsplit("-", 1)[-1])
        except ValueError:
            return -1
    for state_path in sorted(state_paths, key=checkpoint_step, reverse=True):
        try:
            state = json.loads(state_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if _finite_number(state.get("best_metric")):
            return {"validation_loss": float(state["best_metric"]),
                    "validation_loss_source": str(state_path)}

    # Final fallback for adapters produced before durable metric recording was added.
    history_path = output_dir / "trainer_history.csv"
    if history_path.is_file():
        try:
            history = pd.read_csv(history_path)
            losses = (pd.to_numeric(history["eval_loss"], errors="coerce")
                      if "eval_loss" in history.columns else pd.Series(dtype=float))
            losses = losses[np.isfinite(losses)]
        except Exception:
            losses = pd.Series(dtype=float)
        if len(losses):
            return {"validation_loss": float(losses.min()),
                    "validation_loss_source": "trainer_history.csv:min(eval_loss)"}
    return {"validation_loss": None, "validation_loss_source": "unavailable"}


def fold_training_state(output_dir, config):
    """
    Three states, not two. The middle one matters: generation over ~900 held-out images with
    token scoring takes 1-2 hours AFTER training finishes, and an interruption in that window
    used to discard the ~8 GPU-hours of training that had already succeeded, because
    fold_summary.json is only written at the very end.

      "complete" -> summary + adapter present, config matches. Reuse everything.
      "trained"  -> adapter present and config matches, but no summary. Skip TRAINING, reload
                    the adapter, and resume at evaluation.
      "none"     -> train from scratch.
    """
    output_dir = Path(output_dir)
    summary_path = output_dir / "fold_summary.json"
    adapter_dir = output_dir / "best_adapter"
    adapter_path = adapter_dir / "adapter_config.json"
    stamp_path = adapter_dir / "training_config.json"
    inner_scores_path = output_dir / "inner_validation_score_records.jsonl"
    adapter_complete = adapter_path.is_file() and adapter_weights_present(adapter_dir)
    prediction_complete = (not RUN_GENERATION_EVAL
                           or (output_dir / "predictions.jsonl").is_file())
    scoring_complete = (not (RUN_GENERATION_EVAL and RUN_TOKEN_SCORING)
                        or ((output_dir / "score_records.jsonl").is_file()
                            and inner_scores_path.is_file()))
    evaluation_complete = prediction_complete and scoring_complete

    if summary_path.is_file() and adapter_complete:
        try:
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
        except Exception:
            summary = None
        if summary and summary.get("config") == config:
            recovered = recover_training_metrics(output_dir, summary)
            if recovered != {key: summary.get(key) for key in recovered}:
                summary.update(recovered)
                cm.write_json(summary_path, summary)
            if evaluation_complete:
                return "complete", summary
            print("    training is complete, but evaluation/token-score artifacts are "
                  "incomplete; reusing the adapter and resuming evaluation.")
            return "trained", summary
        print("    existing summary was trained on a DIFFERENT config; retraining.")
        return "none", None

    if adapter_complete and stamp_path.is_file():
        try:
            stamped = json.loads(stamp_path.read_text(encoding="utf-8"))
        except Exception:
            stamped = None
        if stamped == config:
            return "trained", None
        print("    saved adapter was trained on a DIFFERENT config; retraining.")
    return "none", None


def train_fold(fold, config, tag, output_dir, evaluate=True, eval_max_images=None,
               save_adapter=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Skip a fold that is already finished under an identical config. Without this, a crash at
    # fold 3 throws away the ~24 GPU-hours already spent on folds 0-2.
    state, existing = fold_training_state(output_dir, config)
    if state == "complete":
        print(f"    fold {fold} already complete under this config; reusing "
              f"{output_dir / 'fold_summary.json'}")
        rows, scores = [], []
        predictions_path = output_dir / "predictions.jsonl"
        if predictions_path.is_file():
            rows = cm.read_jsonl(predictions_path)
        scores_path = output_dir / "score_records.jsonl"
        if scores_path.is_file():
            scores = cm.read_jsonl(scores_path)
        return existing, rows, scores

    optimisation, validation, test = prepare_fold(fold, config)

    if state == "trained":
        # Training already succeeded; only the evaluation was lost. Reload the adapter rather
        # than spending another ~8 GPU-hours reproducing a result that is already on disk.
        print(f"    fold {fold}: adapter present for this config -- skipping training and "
              "resuming at evaluation.")
        from peft import PeftModel
        base = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID, **model_load_kwargs(torch.bfloat16, device_map="auto",
                                          low_cpu_mem_usage=True,
                                          **({"revision": MODEL_REVISION} if MODEL_REVISION else {})))
        base.config.use_cache = True
        model = PeftModel.from_pretrained(
            base, str(output_dir / "best_adapter"), is_trainable=False).eval()
        rows, scores, validation_scores = [], [], []
        if evaluate and RUN_GENERATION_EVAL:
            rows, scores = evaluate_fold(model, test, fold, tag,
                                         max_images=eval_max_images,
                                         checkpoint_dir=output_dir)
            if RUN_TOKEN_SCORING:
                validation_covid = [r for r in validation
                                    if r["task"] == "covid_classification"]
                _, validation_scores = evaluate_fold(
                    model, validation_covid, fold, f"{tag}_inner_validation",
                    max_images=eval_max_images,
                    checkpoint_dir=output_dir / "inner_validation")
        recovered_metrics = recover_training_metrics(output_dir, existing)
        summary = {
            **(existing or {}),
            "fold": fold, "tag": tag, "config": config,
            "lora": json.loads((output_dir / "best_adapter" / "training_stats.json").read_text())
            if (output_dir / "best_adapter" / "training_stats.json").is_file() else {},
            "counts": {"optimisation": len(optimisation), "validation": len(validation),
                       "test": len(test)},
            "resumed_from_saved_adapter": True,
            "adapter_dir": str(output_dir / "best_adapter"),
            **recovered_metrics,
        }
        if rows:
            summary["generation_metrics"] = evaluate_arm(rows, tag)
            cm.write_jsonl(output_dir / "predictions.jsonl", rows)
        if scores:
            cm.write_jsonl(output_dir / "score_records.jsonl", scores)
        if validation_scores:
            cm.write_jsonl(output_dir / "inner_validation_score_records.jsonl",
                           validation_scores)
            summary["inner_validation_covid_n"] = len(validation_scores)
        cm.write_json(output_dir / "fold_summary.json", summary)
        release(model)
        return summary, rows, scores

    print(f"    records: optimise={len(optimisation):,} validate={len(validation):,} "
          f"test={len(test):,}")
    torch.manual_seed(SEED + fold)
    model, lora_stats = build_lora_model(config)

    trainer_dir = output_dir / "trainer"
    # Sweeps only need the validation loss for ranking, so they do not persist checkpoints.
    # 24 sweep configs x two full 4B checkpoints each would waste tens of GB and a lot of I/O.
    keep_checkpoints = save_adapter
    arguments = build_training_arguments(
        output_dir=str(trainer_dir),
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=config["learning_rate"],
        warmup_ratio=config["warmup_ratio"],
        lr_scheduler_type="cosine",
        bf16=True, logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch" if keep_checkpoints else "no",
        save_total_limit=2 if keep_checkpoints else None,
        load_best_model_at_end=bool(keep_checkpoints),
        metric_for_best_model="eval_loss", greater_is_better=False,
        dataloader_num_workers=(DATALOADER_NUM_WORKERS
                                if (OVERRIDE_OLD_KERNEL_GUARD
                                    or SAFE_DATALOADER_WORKERS is None)
                                else SAFE_DATALOADER_WORKERS),
        dataloader_pin_memory=(True if OVERRIDE_OLD_KERNEL_GUARD
                               else SAFE_PIN_MEMORY),
        remove_unused_columns=False, report_to=[], seed=SEED + fold,
        gradient_checkpointing=False,   # already enabled on the base model
    )
    trainer = Trainer(
        model=model, args=arguments,
        train_dataset=HarmonyImageTextDataset(optimisation, processor, MAX_LENGTH),
        eval_dataset=HarmonyImageTextDataset(validation, processor, MAX_LENGTH),
        data_collator=MultiModalResponseOnlyCollator(processor),
    )

    # Resume from the last checkpoint if this fold was interrupted mid-training.
    resume_from = None
    if keep_checkpoints and trainer_dir.is_dir():
        try:
            resume_from = get_last_checkpoint(str(trainer_dir))
        except Exception:
            resume_from = None
        if resume_from:
            print(f"    resuming from {Path(resume_from).name}")

    started = time.perf_counter()
    train_result = trainer.train(resume_from_checkpoint=resume_from)
    train_seconds = time.perf_counter() - started
    validation_metrics = trainer.evaluate()
    durable_training_metrics = {
        "train_loss": float(train_result.training_loss),
        "validation_loss": float(validation_metrics.get("eval_loss", float("nan"))),
        "best_metric": (float(getattr(trainer.state, "best_metric", float("nan")))
                        if _finite_number(getattr(trainer.state, "best_metric", None))
                        else None),
        "train_seconds": round(train_seconds, 1),
    }

    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(output_dir / "trainer_history.csv", index=False)

    adapter_dir = output_dir / "best_adapter"
    if save_adapter:
        trainer.model.save_pretrained(str(adapter_dir))
        processor.save_pretrained(str(adapter_dir))
        # Persist losses before generation, so adapter-only recovery cannot lose the evidence
        # needed by the final stability gate.
        cm.write_json(output_dir / "training_metrics.json", durable_training_metrics)
        # Written immediately after the adapter so that an interruption during the evaluation
        # that follows still leaves enough on disk to prove the training was valid.
        cm.write_json(adapter_dir / "training_config.json", config)
        cm.write_json(adapter_dir / "training_stats.json", lora_stats)

    summary = {
        "fold": fold, "tag": tag, "config": config, "lora": lora_stats,
        "counts": {"optimisation": len(optimisation), "validation": len(validation),
                   "test": len(test)},
        "train_seconds": round(train_seconds, 1),
        "train_loss": float(train_result.training_loss),
        "validation_loss": durable_training_metrics["validation_loss"],
        "validation_loss_source": "training_metrics.json",
        "peak_memory_gib": (round(torch.cuda.max_memory_allocated() / 1024 ** 3, 3)
                            if torch.cuda.is_available() else None),
        "adapter_dir": str(adapter_dir),
    }

    rows, scores, validation_scores = [], [], []
    if evaluate and RUN_GENERATION_EVAL:
        model.config.use_cache = True
        model.eval()
        print("    generating held-out predictions ...")
        rows, scores = evaluate_fold(model, test, fold, tag,
                                     max_images=eval_max_images,
                                     checkpoint_dir=output_dir)
        summary["generation_metrics"] = evaluate_arm(rows, tag)
        if RUN_TOKEN_SCORING:
            validation_covid = [r for r in validation
                                if r["task"] == "covid_classification"]
            _, validation_scores = evaluate_fold(
                model, validation_covid, fold, f"{tag}_inner_validation",
                max_images=eval_max_images,
                checkpoint_dir=output_dir / "inner_validation")

    # Persist per-fold predictions so a completed fold can be reused without regenerating.
    if rows:
        cm.write_jsonl(output_dir / "predictions.jsonl", rows)
    if scores:
        cm.write_jsonl(output_dir / "score_records.jsonl", scores)
    if validation_scores:
        cm.write_jsonl(output_dir / "inner_validation_score_records.jsonl",
                       validation_scores)
        summary["inner_validation_covid_n"] = len(validation_scores)
    if save_adapter:
        cm.write_json(output_dir / "fold_summary.json", summary)
    release(model)
    release(trainer)
    return summary, rows, scores

## 10. Phase A — sweeps on fold 0

Each family varies one thing against the baseline. Ranking is by **inner-validation loss**, so
no test fold influences any hyperparameter choice.

Short runs (`SWEEP_EPOCHS = 2`) are used to rank configurations. That is a deliberate
approximation: it assumes ranking at 2 epochs predicts ranking at 5, which is usually but not
always true. `E5-L` also reports full learning curves so the assumption is visible rather than
hidden, and any family whose top two entries are within noise is reported as "no evidence of a
difference" rather than a winner.

In [ ]:
def config_with(**overrides):
    config = dict(BASE_CONFIG)
    config.update(overrides)
    config.setdefault("lora_alpha", 2 * config["lora_r"])
    return config


SWEEP_LOG = SWEEP_DIR / "sweep_results.jsonl"

# Each sweep config is ~1-2 GPU-hours, so ~24 of them is ~25 GPU-hours. Appending each result
# as it completes means an interrupted sweep resumes instead of restarting.
sweep_results = cm.read_jsonl(SWEEP_LOG) if SWEEP_LOG.is_file() else []
if sweep_results:
    print(f"Resuming sweeps: {len(sweep_results)} config(s) already recorded in "
          f"{SWEEP_LOG.name}")
_sweep_done = {(row.get("family"), row.get("index")) for row in sweep_results}

if RUN_SWEEPS:
    for family in RUN_SWEEP_FAMILIES:
        print("=" * 78)
        print("SWEEP", family)
        for index, overrides in enumerate(SWEEPS[family]):
            if (family, index) in _sweep_done:
                print(f"  [{index + 1}/{len(SWEEPS[family])}] already done; skipping")
                continue
            config = config_with(**overrides, epochs=SWEEP_EPOCHS)
            tag = f"{family}#{index}"
            label = ", ".join(f"{k}={v}" for k, v in overrides.items())
            print(f"  [{index + 1}/{len(SWEEPS[family])}] {label}")
            try:
                summary, _, _ = train_fold(
                    SWEEP_FOLD, config, tag,
                    SWEEP_DIR / family / f"cfg{index}", evaluate=False,
                    save_adapter=False)
            except Exception as exc:
                print(f"    FAILED: {type(exc).__name__}: {exc}")
                row = {"family": family, "index": index, **overrides,
                       "validation_loss": float("nan"),
                       "error": f"{type(exc).__name__}: {exc}"}
                cm.append_jsonl(SWEEP_LOG, row)
                sweep_results.append(row)
                continue
            row = {
                "family": family, "index": index, **overrides,
                "validation_loss": summary["validation_loss"],
                "train_loss": summary["train_loss"],
                "trainable_parameters": summary["lora"]["trainable_parameters"],
                "train_seconds": summary["train_seconds"],
                "peak_memory_gib": summary["peak_memory_gib"],
                "error": None,
            }
            cm.append_jsonl(SWEEP_LOG, row)   # durable before the next config starts
            sweep_results.append(row)
            print(f"    val_loss={summary['validation_loss']:.5f} "
                  f"trainable={summary['lora']['trainable_parameters'] / 1e6:.2f}M "
                  f"{summary['train_seconds']:.0f}s")

    sweep_frame = pd.DataFrame(sweep_results)
    sweep_frame.to_csv(SWEEP_DIR / "all_sweeps.csv", index=False)
    for family in sweep_frame["family"].unique():
        sweep_frame[sweep_frame["family"] == family].to_csv(
            SWEEP_DIR / f"fold0_{family}.csv", index=False)
    print()
    print(sweep_frame[["family", "index", "validation_loss", "trainable_parameters",
                       "train_seconds"]].to_string(index=False))
else:
    sweep_frame = pd.DataFrame(sweep_results) if sweep_results else pd.DataFrame()
    if len(sweep_frame):
        print(f"Sweeps disabled, but {len(sweep_frame)} previously recorded result(s) were "
              "loaded from sweep_results.jsonl and will still inform selection.")
    else:
        print("Sweeps disabled; using BASE_CONFIG for all folds.")

In [ ]:
NOISE_TOLERANCE = 0.002     # inner-validation loss difference treated as indistinguishable

selection = {"base_config": BASE_CONFIG, "chosen": dict(BASE_CONFIG), "families": {}}

if RUN_SWEEPS and len(sweep_frame):
    for family in RUN_SWEEP_FAMILIES:
        subset = sweep_frame[(sweep_frame["family"] == family)
                             & sweep_frame["validation_loss"].notna()]
        if not len(subset):
            selection["families"][family] = {"status": "all runs failed"}
            continue
        ordered = subset.sort_values("validation_loss")
        best = ordered.iloc[0]
        overrides = {key: best[key] for key in SWEEPS[family][int(best["index"])]}
        runner_up = ordered.iloc[1] if len(ordered) > 1 else None
        margin = (float(runner_up["validation_loss"] - best["validation_loss"])
                  if runner_up is not None else float("inf"))
        decisive = margin > NOISE_TOLERANCE

        selection["families"][family] = {
            "best": overrides,
            "best_validation_loss": float(best["validation_loss"]),
            "margin_to_runner_up": margin,
            "decisive": bool(decisive),
            "note": ("adopted" if decisive else
                     "within noise of the runner-up; baseline retained and reported as "
                     "'no evidence of a difference'"),
        }
        if decisive:
            selection["chosen"].update(overrides)
        print(f"{family}: best={overrides} val_loss={best['validation_loss']:.5f} "
              f"margin={margin:.5f} -> {'ADOPT' if decisive else 'keep baseline (noise)'}")

if FORCE_FINAL_CONFIG:
    selection["chosen"].update(FORCE_FINAL_CONFIG)
    selection["forced"] = FORCE_FINAL_CONFIG
    print(f"\nFORCE_FINAL_CONFIG applied: {FORCE_FINAL_CONFIG}")

FINAL_CONFIG = config_with(**{k: v for k, v in selection["chosen"].items()
                              if k in BASE_CONFIG or k == "lora_alpha"})
FINAL_CONFIG["epochs"] = BASE_CONFIG["epochs"]
selection["final_config"] = FINAL_CONFIG
cm.write_json(NB10_DIR / "sweep_selection.json", selection)

print()
print("FINAL CONFIG for the five-fold run:")
print(json.dumps(FINAL_CONFIG, indent=2))
print()
print("Every value above is either the tested default or a sweep winner that beat its "
      "runner-up by more than the noise tolerance. That provenance is what Table S1 reports, "
      "and it is the answer to referee 1.1.")

## 11. Phase B — the selected configuration on all five folds

Each fold reloads a pristine base checkpoint and trains its own adapter. No adapter, threshold,
or selection crosses a fold boundary.

In [ ]:
FINAL_ARM = "A3_qwen_lora_final"
fold_summaries, predictions, score_records, inner_validation_scores = {}, [], [], []

if RUN_FINAL_FOLDS:
    for fold in FINAL_FOLDS:
        print("=" * 78)
        print(f"FINAL FOLD {fold}")
        summary, rows, scores = train_fold(
            fold, FINAL_CONFIG, FINAL_ARM, FOLD_DIR / f"fold_{fold}",
            evaluate=True, eval_max_images=(16 if SMOKE_TEST else None))
        fold_summaries[fold] = summary
        predictions.extend(rows)
        score_records.extend(scores)
        inner_path = FOLD_DIR / f"fold_{fold}" / "inner_validation_score_records.jsonl"
        if inner_path.is_file():
            inner_validation_scores.extend(cm.read_jsonl(inner_path))
        if "generation_metrics" in summary:
            print_arm_summary(summary["generation_metrics"])

    cm.write_jsonl(NB10_DIR / "predictions_qwen_lora.jsonl", predictions)
    if score_records:
        frame = pd.DataFrame(score_records)
        try:
            frame.to_parquet(NB10_DIR / "token_probability_scores.parquet", index=False)
        except Exception:
            frame.to_csv(NB10_DIR / "token_probability_scores.csv", index=False)
    print()
    print(f"Wrote {len(predictions):,} out-of-fold predictions")

## 12. Metrics, threshold tuning (E8d), and the arm summary

E8d is applied here rather than during training: the operating point is chosen on **inner
validation** and then applied unchanged to the test fold. Two points are reported — Youden's J
and a fixed sensitivity of 0.90 — because a screening application and a confirmatory
application want different trade-offs, and the rejected version reported neither.

In [ ]:
pooled = evaluate_arm(predictions, FINAL_ARM) if predictions else {}
per_fold_metrics = {fold: summary["generation_metrics"]
                    for fold, summary in fold_summaries.items()
                    if "generation_metrics" in summary}
aggregate = cm.aggregate_over_folds(per_fold_metrics) if per_fold_metrics else []
if aggregate:
    pd.DataFrame(aggregate).to_csv(
        NB10_DIR / "cross_validation_aggregate_95ci.csv", index=False)

def binary_counts(truth, predicted):
    truth, predicted = np.asarray(truth), np.asarray(predicted)
    tp = int(((predicted == 1) & (truth == 1)).sum())
    tn = int(((predicted == 0) & (truth == 0)).sum())
    fp = int(((predicted == 1) & (truth == 0)).sum())
    fn = int(((predicted == 0) & (truth == 1)).sum())
    sensitivity = tp / (tp + fn) if tp + fn else None
    specificity = tn / (tn + fp) if tn + fp else None
    balanced = ((sensitivity + specificity) / 2
                if sensitivity is not None and specificity is not None else None)
    return {"sensitivity": sensitivity, "specificity": specificity,
            "balanced_accuracy": balanced, "tp": tp, "tn": tn,
            "fp": fp, "fn": fn}


def select_inner_threshold(rows, target_sensitivity=None):
    truth = np.array([1 if row["gt_covid"] == "Yes" else 0 for row in rows])
    score = np.array([float(row["covid_score"]) for row in rows])
    if set(truth.tolist()) != {0, 1}:
        raise RuntimeError("Inner validation must contain both COVID classes.")
    candidates = np.unique(np.r_[0.0, np.round(score, 6), 1.0])
    ranked = []
    for threshold in candidates:
        stats = binary_counts(truth, (score >= threshold).astype(int))
        if target_sensitivity is None:
            value = stats["sensitivity"] + stats["specificity"] - 1
        elif stats["sensitivity"] >= target_sensitivity:
            value = stats["specificity"]
        else:
            continue
        ranked.append((value, -abs(float(threshold) - 0.5), float(threshold)))
    if not ranked:
        raise RuntimeError("No valid inner-validation operating threshold.")
    return max(ranked)[2]


threshold_rows, pooled_operating = [], defaultdict(list)
for fold in FINAL_FOLDS:
    validation = [row for row in inner_validation_scores
                  if int(row["fold"]) == fold
                  and row.get("covid_score") is not None
                  and row.get("gt_covid") is not None]
    test = [row for row in predictions
            if int(row["held_out_fold"]) == fold
            and row.get("covid_score") is not None
            and row.get("gt_covid") is not None]
    if not validation or not test:
        continue
    test_truth = np.array([1 if row["gt_covid"] == "Yes" else 0 for row in test])
    test_score = np.array([float(row["covid_score"]) for row in test])
    for name, target in [("youden_j", None), ("sensitivity_0.90", 0.90)]:
        threshold = select_inner_threshold(validation, target)
        decisions = (test_score >= threshold).astype(int)
        stats = binary_counts(test_truth, decisions)
        threshold_rows.append({"fold": fold, "operating_point": name,
            "threshold": round(threshold, 6), "n_validation": len(validation),
            "n_test": len(test),
            **{key: (round(value, 4) if isinstance(value, float) else value)
               for key, value in stats.items()},
            "threshold_source": "inner_validation",
            "evaluation_source": "outer_test"})
        pooled_operating[name].extend(zip(test_truth.tolist(), decisions.tolist()))

for name, pairs in pooled_operating.items():
    truth, decisions = map(np.asarray, zip(*pairs))
    stats = binary_counts(truth, decisions)
    threshold_rows.append({"fold": "pooled_oof", "operating_point": name,
        "threshold": None, "n_validation": None, "n_test": len(pairs),
        **{key: (round(value, 4) if isinstance(value, float) else value)
           for key, value in stats.items()},
        "threshold_source": "per_fold_inner_validation",
        "evaluation_source": "pooled_outer_test"})

if threshold_rows:
    pd.DataFrame(threshold_rows).to_csv(NB10_DIR / "operating_points.csv", index=False)
    print("E8d operating points (thresholds selected on inner validation):")
    print(pd.DataFrame(threshold_rows).to_string(index=False))
elif RUN_FINAL_FOLDS and RUN_GENERATION_EVAL and RUN_TOKEN_SCORING:
    raise RuntimeError("No inner-validation COVID scores; operating points are unavailable.")


def fold_ci(metric):
    match = [row for row in aggregate if row["metric"] == metric]
    return ((match[0]["mean"], match[0]["ci95_lower"], match[0]["ci95_upper"])
            if match else (None, None, None))


covid, mrale = pooled.get("covid", {}), pooled.get("mrale", {})
mae_mean, mae_low, mae_high = fold_ci("mrale.mae")
auroc_mean, auroc_low, auroc_high = fold_ci("covid.auroc")
lora_stats = next(iter(fold_summaries.values()))["lora"] if fold_summaries else {}

summary_row = OrderedDict([
    ("arm", FINAL_ARM), ("agent", AGENT_NAME), ("model_id", MODEL_ID),
    ("adaptation", f"LoRA r={FINAL_CONFIG['lora_r']} {FINAL_CONFIG['target_policy']}"),
    ("trainable_parameters_M", round(lora_stats.get("trainable_parameters", 0) / 1e6, 2)),
    ("n_images", pooled.get("n_rows", 0)),
    ("mrale_mae_pooled", round(mrale.get("mae", float("nan")), 3)),
    ("mrale_mae_ci95", None if mae_mean is None else f"[{mae_low:.3f}, {mae_high:.3f}]"),
    ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
    ("covid_auroc_ci95", None if auroc_mean is None else f"[{auroc_low:.4f}, {auroc_high:.4f}]"),
    ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
    ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
    ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
    ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
    ("covid_f1", round(covid.get("f1", float("nan")), 4)),
    ("covid_mcc", round(covid.get("mcc", float("nan")), 4)),
    ("covid_brier", round(covid.get("brier", float("nan")), 4)),
    ("covid_ece", round(covid.get("ece", float("nan")), 4)),
    ("mrale_rmse", round(mrale.get("rmse", float("nan")), 3)),
    ("mrale_qwk", round(mrale.get("qwk", float("nan")), 4)),
    ("mrale_spearman", round(mrale.get("spearman_rho", float("nan")), 4)),
    ("mrale_within1", round(mrale.get("within1_accuracy", float("nan")), 4)),
    ("mrale_coverage", round(mrale.get("coverage", float("nan")), 4)),
    ("mrale_formula_consistency", round(mrale.get("formula_consistency", float("nan")), 4)),
    ("mae_band_none", round(mrale.get("mae_band_none", float("nan")), 3)),
    ("mae_band_mild", round(mrale.get("mae_band_mild", float("nan")), 3)),
    ("mae_band_moderate", round(mrale.get("mae_band_moderate", float("nan")), 3)),
    ("mae_band_severe", round(mrale.get("mae_band_severe", float("nan")), 3)),
])
pd.DataFrame([summary_row]).to_csv(NB10_DIR / "arm_summary.csv", index=False)
cm.write_json(NB10_DIR / "per_fold_metrics.json", per_fold_metrics)

pd.set_option("display.width", 220)
print()
print(pd.DataFrame([summary_row]).T.to_string())
print()
print("=" * 78)
print("GATE G3 -- NEW REFERENCE POINT (restated; see the notebook header)")
print(f"  clean-fold mRALE MAE          : {mrale.get('mae', float('nan')):.3f}")
print(f"  superseded leaky-fold MAE     : {SUPERSEDED_LEAKY_REFERENCE['mrale_mae']:.3f}")
print(f"  clean-fold COVID balanced acc : {covid.get('balanced_accuracy', float('nan')):.4f}")
print(f"  superseded leaky-fold bal acc : "
      f"{SUPERSEDED_LEAKY_REFERENCE['covid_balanced_accuracy']:.4f}")
print()
print("  The old figures came from folds with 135-153 study groups on both sides of the")
print("  split. They are NOT a target to reproduce. Degradation here is the expected and")
print("  correct outcome; quote both numbers in the response letter as evidence that the")
print("  leak was found and fixed rather than discovered in review.")

## 13. External cohorts

In [ ]:
external_predictions = []
if RUN_EXTERNAL and fold_summaries:
    external_dir = NB03_DIR / "external_manifests"
    external_records = []
    if external_dir.is_dir():
        for path in sorted(external_dir.glob("*_harmony.jsonl")):
            external_records.extend(read_jsonl(path))
    external_records = [record for record in external_records
                        if record.get("task") in DIRECT_TASKS]

    if external_records:
        internal_names = {record["file_name"] for fold in range(N_FOLDS)
                          for split in ["train", "test"]
                          for record in fold_records[fold][split]}
        leaked = {record["file_name"] for record in external_records} & internal_names
        if leaked:
            raise RuntimeError(
                f"{len(leaked)} external filenames occur in the internal folds "
                f"(e.g. {sorted(leaked)[:3]}). Five-adapter ensembling on these would be a "
                "leakage result. Re-run NB 03's membership guard.")

        print(f"External records: {len(external_records):,} "
              f"(leakage check passed on {len(internal_names):,} internal filenames)")
        from peft import PeftModel
        aggregated = defaultdict(list)
        for fold in FINAL_FOLDS:
            adapter_dir = Path(fold_summaries[fold]["adapter_dir"])
            base = AutoModelForMultimodalLM.from_pretrained(
                MODEL_ID, **model_load_kwargs(
                    torch.bfloat16, device_map="auto", low_cpu_mem_usage=True,
                    **({"revision": MODEL_REVISION} if MODEL_REVISION else {})))
            base.config.use_cache = True
            model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False).eval()
            rows, _ = evaluate_fold(model, external_records, None,
                                    f"{FINAL_ARM}_external_fold{fold}")
            for row in rows:
                aggregated[row["image_key"]].append(row)
            release(model)
            print(f"  fold {fold} adapter: {len(rows)} external predictions")

        for image_key, rows in aggregated.items():
            totals = [row["mrale_total"] for row in rows if row.get("mrale_total") is not None]
            scores = [row["covid_score"] for row in rows if row.get("covid_score") is not None]
            template = rows[0]
            mean_score = float(np.mean(scores)) if scores else None
            external_predictions.append(cm.make_prediction_row(
                image_key=image_key, cohort=template["cohort"],
                subcohort=template["subcohort"], filename=template["filename"],
                held_out_fold=None, agent=AGENT_NAME, arm=f"{FINAL_ARM}_external",
                view="v0", task="joint",
                covid_pred=(None if mean_score is None else
                            ("Yes" if mean_score >= 0.5 else "No")),
                covid_score=mean_score,
                mrale_total=int(round(float(np.median(totals)))) if totals else None,
                valid=bool(totals or scores), parse_error=None,
                model_id=MODEL_ID, model_revision=MODEL_REVISION,
                ensemble_of_folds=len(rows),
                **{key: template.get(key) for key in
                   ["gt_covid", "gt_mrale_total", "gt_mrale_right", "gt_mrale_left"]},
            ))
        cm.write_jsonl(NB10_DIR / "external_predictions.jsonl", external_predictions)

        rows_by_subcohort = defaultdict(list)
        for row in external_predictions:
            rows_by_subcohort[row["subcohort"]].append(row)
        external_summary = []
        for subcohort, rows in sorted(rows_by_subcohort.items()):
            covid_metrics = evaluate_arm(rows, FINAL_ARM).get("covid", {})
            totals = [row["mrale_total"] for row in rows if row.get("mrale_total") is not None]
            external_summary.append({
                "subcohort": subcohort, "n": len(rows),
                "specificity": round(covid_metrics.get("specificity", float("nan")), 4),
                "mean_predicted_mrale": round(float(np.mean(totals)), 2) if totals else None,
            })
        frame = pd.DataFrame(external_summary)
        frame.to_csv(NB10_DIR / "external_summary.csv", index=False)
        print()
        print(frame.to_string(index=False))
else:
    print("External evaluation skipped.")

## 14. Run configuration and gate

In [ ]:
score_validation = []
if score_records:
    frame = pd.DataFrame(score_records)
    frame = frame[frame["generated_decision"].notna()]
    if len(frame):
        agreement = float((frame["scored_decision"] == frame["generated_decision"]).mean())
        score_validation.append({
            "arm": FINAL_ARM, "n_scored": len(frame),
            "score_generation_agreement": round(agreement, 5),
            "meets_protocol_7_2": bool(agreement >= 0.995),
        })
        print(f"Token-score validation: agreement={agreement:.5f} "
              f"({'PASS' if agreement >= 0.995 else 'FAIL'})")

integrity = {}
if predictions:
    objects = [value for row in predictions for key, value in row.get("extra", {}).items()
               if key.endswith("n_json_objects")]
    integrity = {
        "duplicate_object_rate": round(float(np.mean([n > 1 for n in objects])), 4)
        if objects else None,
        "mean_json_objects": round(float(np.mean(objects)), 3) if objects else None,
        "valid_rate": round(float(np.mean([bool(r["valid"]) for r in predictions])), 4),
        "formula_consistency": round(float(np.mean(
            [bool(r.get("extra", {}).get("formula_consistent"))
             for r in predictions if "formula_consistent" in r.get("extra", {})])), 4)
        if any("formula_consistent" in r.get("extra", {}) for r in predictions) else None,
    }
    print("Output integrity:", json.dumps(integrity, indent=2))

# Repair summaries written by the old adapter-only resume path. This reads Trainer metadata;
# it does not retrain or alter adapter weights.
for fold, summary in fold_summaries.items():
    recovered = recover_training_metrics(FOLD_DIR / f"fold_{fold}", summary)
    summary.update(recovered)
    summary_path = FOLD_DIR / f"fold_{fold}" / "fold_summary.json"
    if summary_path.is_file():
        cm.write_json(summary_path, summary)
    print(f"Fold {fold} validation loss: {summary.get('validation_loss')} "
          f"[{summary.get('validation_loss_source')}]")

cm.write_json(NB10_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "10_qwen35_multitask_lora_5fold.ipynb",
    "protocol_agent": AGENT_NAME,
    "protocol_experiments": ["E5-R", "E5-T", "E5-L", "E5-M", "E8", "E8d"],
    "deferred": {
        "E5-H": "Scalar-regression and ordinal head variants are NOT implemented here. They "
                "require a different loss and evaluation path, and the generative arm is the "
                "one the reasoning framework needs. Implement in a separate notebook if the "
                "accuracy tax of generative output has to be quantified.",
        "E5-S": "NV-Reason LoRA lives in NB 11.",
    },
    "seed": SEED,
    "environment_shims": {
        "kernel": _platform.release(),
        "old_kernel_guard_active": bool(OLD_KERNEL
                                        and not OVERRIDE_OLD_KERNEL_GUARD),
        "dataloader_workers_used": (DATALOADER_NUM_WORKERS
            if (OVERRIDE_OLD_KERNEL_GUARD or SAFE_DATALOADER_WORKERS is None)
            else SAFE_DATALOADER_WORKERS),
        "dtype_kwarg": list(model_load_kwargs().keys())[0],
    },
    "model": {"model_id": MODEL_ID, "revision": MODEL_REVISION},
    "rq1_matching": {
        "inherit_nb09_config": INHERIT_NB09_CONFIG,
        "inherited": INHERITED_FROM_NB09,
        "rationale": (
            "RQ1 is a medical-specialization test only if the two arms differ in pretraining "
            "and not in tuning effort. Sweeping Qwen while MedGemma keeps a configuration "
            "chosen under a shorter schedule would confound the two."
        ),
    },
    "architecture_deviations": {
        "loader_class": "AutoModelForMultimodalLM (Qwen3.5) vs AutoModelForImageTextToText",
        "visual_token_budget": {"min_tokens": MIN_IMAGE_TOKENS, "max_tokens": MAX_IMAGE_TOKENS,
                                "min_pixels": MIN_IMAGE_PIXELS, "max_pixels": MAX_IMAGE_PIXELS,
                                "note": "Qwen3.5 uses dynamic resolution; MedGemma a fixed grid."},
        "lora_targets": ("module scan over the text backbone, because Qwen3.5 mixes full and "
                         "linear attention; a fixed q/k/v/o list would leave much of it "
                         "unadapted"),
        "thinking_mode": {"disabled": DISABLE_THINKING,
                          "note": "Qwen3.5 reasons by default; direct JSON supervision must "
                                  "not contain a <think> trace."},
        "disclosure": ("All four differ from NB 09 by architectural necessity and must be "
                       "stated in the manuscript rather than presented as identical settings. "
                       "Trainable-parameter counts differ; report both."),
    },
    "fold_source": str(FOLD_DATA_DIR),
    "tasks": {"direct": DIRECT_TASKS, "auxiliary": AUXILIARY_TASKS,
              "auto_detected_from": "regenerated fold files"},
    "base_config": BASE_CONFIG,
    "final_config": FINAL_CONFIG,
    "sweep_selection": selection,
    "noise_tolerance": NOISE_TOLERANCE,
    "training": {
        "max_length": MAX_LENGTH,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "scheduler": "cosine", "precision": "bf16",
        "checkpoint_selection": "inner-validation loss",
    },
    "decoding": {"do_sample": False, "json_stop_criteria": USE_JSON_STOP_CRITERIA,
                 "max_new_tokens": GENERATION_MAX_NEW_TOKENS},
    "token_scoring": {"enabled": RUN_TOKEN_SCORING,
                      "method": "teacher-forced sequence log-likelihood, identical to NB 07",
                      "validation": score_validation},
    "output_integrity": integrity,
    "operating_points": threshold_rows,
    "superseded_leaky_reference": SUPERSEDED_LEAKY_REFERENCE,
    "gate_g3_restated": (
        "The protocol's original G3 ('M5 reproduces the existing reference numbers') is not "
        "applied: those numbers came from leaky folds and are invalid. G3 here is that M5 "
        "trains stably on the regenerated folds and establishes the new reference."
    ),
    "fold_summaries": {str(k): {key: value for key, value in v.items()
                                if key != "generation_metrics"}
                       for k, v in fold_summaries.items()},
    "smoke_test": SMOKE_TEST,
})

# Usability flags use the NB 07/08 convention for NB 13 and later consumers.
usability = {FINAL_ARM: {"score_usable": None, "mrale_usable": None}}

failures, warnings = [], []

if RUN_FINAL_FOLDS:
    if set(fold_summaries) != set(FINAL_FOLDS):
        failures.append(f"Only folds {sorted(fold_summaries)} completed of {FINAL_FOLDS}.")
    expected = set()
    for fold in FINAL_FOLDS:
        expected |= {f"MIDRC::{record['file_name']}"
                     for record in fold_records[fold]["test"]
                     if record["task"] in DIRECT_TASKS}
    covered = {row["image_key"] for row in predictions}
    if not SMOKE_TEST and covered != expected:
        failures.append(
            f"Out-of-fold coverage {len(covered)} of {len(expected)}. Every held-out image "
            "must be predicted exactly once, by the fold that excluded it.")
    repeated = [key for key, count in Counter(
        row["image_key"] for row in predictions).items() if count > 1]
    if repeated:
        failures.append(f"{len(repeated)} images predicted more than once.")

    for fold, metrics in per_fold_metrics.items():
        auroc = metrics.get("covid", {}).get("auroc")
        if auroc is None or (isinstance(auroc, float) and math.isnan(auroc)):
            failures.append(f"Fold {fold}: AUROC not computable (endpoint P2).")
    for fold in FINAL_FOLDS:
        summary = fold_summaries.get(fold, {})
        loss = summary.get("validation_loss")
        if not _finite_number(loss):
            failures.append(
                f"Fold {fold}: validation loss is unavailable or non-finite "
                f"(source={summary.get('validation_loss_source', 'unknown')}). Inspect "
                "trainer_history.csv before concluding that training diverged.")

for entry in score_validation:
    if not entry["meets_protocol_7_2"]:
        failures.append(
            f"Token-score agreement {entry['score_generation_agreement']:.4f} < 0.995. "
            "Protocol 7.2 forbids using this score for AUROC/DeLong until resolved.")

if RUN_TOKEN_SCORING and not score_validation:
    failures.append("Token scoring was enabled but produced no validated scores; this arm "
                    "would have no AUROC and could not enter Fig. 3.")

if integrity.get("duplicate_object_rate") and integrity["duplicate_object_rate"] > 0.01:
    warnings.append(f"Duplicate JSON objects at {integrity['duplicate_object_rate']:.3f}; "
                    "the E6-Q stop criteria are not firing everywhere.")

if RUN_SWEEPS:
    indecisive = [family for family, entry in selection["families"].items()
                  if not entry.get("decisive", False)]
    if indecisive:
        warnings.append(
            f"Sweep families with no decisive winner: {indecisive}. Report these as 'no "
            "evidence of a difference' in Table 6 rather than naming a winner -- an "
            "exploratory sweep that lands within noise is a finding, not a failure.")
    failed = [row for row in sweep_results if row.get("error")]
    if failed:
        warnings.append(f"{len(failed)} sweep runs errored; see all_sweeps.csv.")
else:
    warnings.append("Sweeps were skipped, so Table S1 has no empirical provenance for rank, "
                    "target modules, LR, or epochs. Referee 1.1 asked for exactly that.")

if pooled:
    coverage = pooled.get("mrale", {}).get("coverage", float("nan"))
    if not math.isnan(coverage) and coverage < 0.95:
        warnings.append(f"mRALE coverage {coverage:.3f}: the penalised MAE is partly driven "
                        "by the 24-point invalid penalty. Report coverage beside every MAE.")
    specificity = pooled.get("covid", {}).get("specificity", float("nan"))
    if not math.isnan(specificity) and specificity < 0.5:
        warnings.append(
            f"COVID specificity {specificity:.3f} at the default 0.5 threshold. This is the "
            "prevalence-prior problem from the rejected version. operating_points.csv gives "
            "tuned alternatives; E8 arms in the sweep show whether training-side fixes help.")

if SMOKE_TEST:
    warnings.append("SMOKE TEST MODE: nothing here is quotable.")

# ---- usability flags + Table 2 stamping --------------------------------------------------
score_ok = bool(score_validation and all(e["meets_protocol_7_2"] for e in score_validation))
coverage = pooled.get("mrale", {}).get("coverage", float("nan")) if pooled else float("nan")
mrale_ok = bool(not math.isnan(coverage) and coverage > 0.0)
usability[FINAL_ARM].update({
    "score_usable": score_ok, "mrale_usable": mrale_ok,
    "mrale_coverage": None if math.isnan(coverage) else round(coverage, 4),
    "score_agreement": (score_validation[0].get("score_generation_agreement")
                        if score_validation else None),
})
cm.write_json(NB10_DIR / "usability.json", usability)

summary_path = NB10_DIR / "arm_summary.csv"
if summary_path.is_file():
    stamped = pd.read_csv(summary_path)
    stamped["score_usable"] = score_ok
    stamped["mrale_usable"] = mrale_ok
    stamped["report_in_table2"] = (
        "full row" if (score_ok and mrale_ok)
        else "omit AUROC (score quarantined)" if mrale_ok
        else "omit mRALE (coverage 0)" if score_ok
        else "omit mRALE; omit AUROC")
    stamped.to_csv(summary_path, index=False)
    print("arm_summary.csv stamped:",
          {k: stamped.iloc[0][k] for k in
           ["arm", "score_usable", "mrale_usable", "report_in_table2"]})

# ---- RQ1: medical specialization at matched scale ----------------------------------------
FROZEN_PROBE_REFERENCE_MAE = 3.905      # NB 08 frozen BiomedCLIP linear probe, same folds
rq1_rows = []
nb09_summary = NB09_DIR / "arm_summary.csv"
if pooled and nb09_summary.is_file():
    medgemma = pd.read_csv(nb09_summary).iloc[0].to_dict()
    qwen_mae = pooled.get("mrale", {}).get("mae", float("nan"))
    qwen_auroc = pooled.get("covid", {}).get("auroc", float("nan"))
    medgemma_mae = float(medgemma.get("mrale_mae_pooled", float("nan")))
    medgemma_auroc = float(medgemma.get("covid_auroc_pooled", float("nan")))
    qwen_params = lora_stats.get("trainable_parameters", 0) / 1e6
    medgemma_params = float(medgemma.get("trainable_parameters_M", float("nan")))

    rq1_rows = [
        {"arm": "A2_medgemma_lora (medical)", "mrale_mae": round(medgemma_mae, 3),
         "covid_auroc": round(medgemma_auroc, 4),
         "trainable_M": round(medgemma_params, 2)},
        {"arm": "A3_qwen_lora (general)", "mrale_mae": round(qwen_mae, 3),
         "covid_auroc": round(qwen_auroc, 4), "trainable_M": round(qwen_params, 2)},
        {"arm": "A6_biomedclip frozen probe (NB 08)",
         "mrale_mae": FROZEN_PROBE_REFERENCE_MAE, "covid_auroc": None,
         "trainable_M": 0.01},
    ]
    pd.DataFrame(rq1_rows).to_csv(NB10_DIR / "rq1_comparison.csv", index=False)
    print()
    print("RQ1 -- medical specialization at matched scale")
    print(pd.DataFrame(rq1_rows).to_string(index=False))

    mae_delta = qwen_mae - medgemma_mae
    parameter_ratio = (qwen_params / medgemma_params
                       if medgemma_params and not math.isnan(medgemma_params) else float("nan"))
    direction = ("MedGemma better" if mae_delta > 0.1 else
                 "Qwen better" if mae_delta < -0.1 else "no difference beyond 0.1 MAE")
    warnings.append(
        f"RQ1: mRALE MAE MedGemma {medgemma_mae:.3f} vs Qwen {qwen_mae:.3f} "
        f"(delta {mae_delta:+.3f}) -> {direction}. Trainable parameters "
        f"{medgemma_params:.1f}M vs {qwen_params:.1f}M (ratio {parameter_ratio:.2f}x). "
        "Interpret with the parameter ratio in view: a win carrying substantially more "
        "trainable parameters is a scale result, not a specialization result, and the four "
        "architecture deviations in run_config.json must be disclosed alongside it."
    )
    if not math.isnan(qwen_mae) and qwen_mae > FROZEN_PROBE_REFERENCE_MAE - 0.1:
        warnings.append(
            f"GATE G2: this 4B LoRA arm (MAE {qwen_mae:.3f}) does not clearly beat the frozen "
            f"BiomedCLIP linear probe ({FROZEN_PROBE_REFERENCE_MAE:.3f}) from NB 08. If that "
            "holds for both LoRA arms, the paper's framing should move to the decomposition "
            "and reasoning machinery rather than backbone scale."
        )
elif pooled:
    warnings.append(
        f"{nb09_summary} not found, so the RQ1 comparison table was not built. That is expected "
        "if NB 09 is still running: its arm_summary.csv appears only when NB 09 FINISHES, "
        "whereas this notebook needed only its Phase A selection in order to start. Re-run this "
        "notebook once NB 09 completes -- every fold is cached, so it skips to the metrics and "
        "emits rq1_comparison.csv in minutes."
    )

def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB10_DIR / "gate_nb10.json", {
    "passed": not failures, "failures": failures, "warnings": warnings,
    "final_config": FINAL_CONFIG,
    "new_reference": {
        "mrale_mae": pooled.get("mrale", {}).get("mae"),
        "covid_auroc": pooled.get("covid", {}).get("auroc"),
        "covid_balanced_accuracy": pooled.get("covid", {}).get("balanced_accuracy"),
    },
    "superseded_leaky_reference": SUPERSEDED_LEAKY_REFERENCE,
})
# Reasons go in the message so a pasted traceback is self-explanatory.
if failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(failures))
    raise AssertionError(
        f"NB 10 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 10 gate: PASSED")

## Notes carried forward

- **`sweep_selection.json` is Table S1.** Every hyperparameter is labelled as either the tested
  default or a sweep winner that beat its runner-up by more than the noise tolerance. Families
  that landed within noise are reported as "no evidence of a difference" — that is an honest
  answer to referee 1.1, and a more defensible one than naming a winner from a 0.0004 loss gap.
- **E5-H is deliberately not implemented here.** Scalar-regression and ordinal head variants
  need a different loss and evaluation path, and the generative arm is the one the reasoning
  framework actually consumes. If the accuracy tax of generative output has to be quantified,
  that belongs in its own notebook rather than bolted onto this one.
- **The G3 restatement matters for the response letter.** Do not present the clean-fold numbers
  as an improvement or apologise for them being worse. The correct framing is: we found a
  study-level leak in our own folds, regenerated them, and are reporting the corrected figures.
  That is a strength in a revision, not a weakness.
- Token scoring here is byte-identical to NB 07's, so zero-shot and LoRA AUROCs are comparable
  and DeLong in NB 17 is valid across them.
- NB 10 (Qwen3.5) must reuse `FINAL_CONFIG`'s rank, LR, and schedule where the architectures
  permit, and document every place they do not. RQ1 is only a medical-specialization test if
  the two arms are matched.